# Setup and load all models from models/ folder

In [ ]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

model_names = ["RW", "DNS", "Ridge", "XGBoost"]

model_preds = {}
model_actuals = {}
model_metrics = {}

for name in model_names:
    path = MODEL_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        bundle = pickle.load(f)

    print(f"Loaded {name} from {path}, keys: {list(bundle.keys())}")

    model_preds[name]   = bundle["predictions"]
    model_actuals[name] = bundle["actuals"]
    model_metrics[name] = bundle["metrics"]   # <--- NEW

print("Models loaded:", list(model_preds.keys()))


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------- Paths -----------------
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# ----------------- Load raw FRED data -----------------
DGS1 = pd.read_csv(DATA_DIR / "DGS1.csv")
DGS2 = pd.read_csv(DATA_DIR / "DGS2.csv")
DGS5 = pd.read_csv(DATA_DIR / "DGS5.csv")
DGS10 = pd.read_csv(DATA_DIR / "DGS10.csv")

# 2y, 5y, 10y panel
merged = (
    DGS2.merge(DGS5, on="observation_date", how="inner")
        .merge(DGS10, on="observation_date", how="inner")
)
merged = merged.dropna(subset=["DGS2", "DGS5", "DGS10"])
merged = merged.set_index("observation_date")
merged.index = pd.to_datetime(merged.index, errors="coerce")

# Short rate (1y) as separate series
DGS1 = DGS1.dropna(subset=["DGS1"])
DGS1 = DGS1.set_index("observation_date")
DGS1.index = pd.to_datetime(DGS1.index, errors="coerce")

short_rate = DGS1["DGS1"]   # <-- THIS is what you pass to econ functions

# ----------------- Settings -----------------
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]
idx = merged.index
train_start = pd.Timestamp("2009-01-02")
train_end   = pd.Timestamp("2018-12-31")

test_start  = pd.Timestamp("2019-01-02")
test_end    = pd.Timestamp("2025-11-25")  


# Robustness Check of RMSE

In [ ]:
# =====================================================
# RMSE robustness: half–half split of the OOS period
# Split date chosen to equalize sample sizes (by number of OOS dates)
# =====================================================

def rmse(a, f):
    a = np.asarray(a)
    f = np.asarray(f)
    return np.sqrt(np.mean((a - f) ** 2))

# ---- Build OOS date index and pick split date to equalize counts ----
oos_idx = idx[(idx >= test_start) & (idx <= test_end)].sort_values()
if len(oos_idx) < 10:
    raise ValueError("OOS index seems too short — check idx/test_start/test_end.")

mid_pos = len(oos_idx) // 2
split_date = oos_idx[mid_pos]  # equal-count split point (mid-2022-ish in your sample)
print(f"Equal-count OOS split date: {split_date.date()}  "
      f"(N1={mid_pos}, N2={len(oos_idx)-mid_pos}, N_total={len(oos_idx)})")

# ---- Helper to align and compute RMSE over a mask of target dates ----
def rmse_over_period(y_true, y_pred, date_mask):
    # y_true and y_pred are pd.Series with DateTimeIndex
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if df.empty:
        return np.nan, 0
    df = df.loc[df.index.intersection(date_mask)]
    df = df.dropna()
    if df.empty:
        return np.nan, 0
    return rmse(df["y_true"], df["y_pred"]), len(df)

# ---- Compute RMSEs ----
rows = []

# Define the three evaluation date sets (based on the target date t+h)
dates_full = oos_idx
dates_1 = oos_idx[oos_idx < split_date]
dates_2 = oos_idx[oos_idx >= split_date]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # actuals are the realized y_{t+h} aligned to the forecast evaluation date (your bundle design)
        # We use RW actuals as canonical since they should be identical across models
        if key not in model_actuals["RW"]:
            continue
        y_true_all = model_actuals["RW"][key]

        for model in model_names:
            if key not in model_preds[model]:
                continue
            y_pred_all = model_preds[model][key]

            # Align to OOS and compute period RMSEs
            rmse_full, n_full = rmse_over_period(y_true_all, y_pred_all, dates_full)
            rmse_1, n_1 = rmse_over_period(y_true_all, y_pred_all, dates_1)
            rmse_2, n_2 = rmse_over_period(y_true_all, y_pred_all, dates_2)

            rows.append({
                "Maturity": maturity,
                "Horizon": h,
                "Model": model,
                "RMSE_Full": rmse_full,
                "RMSE_2019_to_split": rmse_1,
                "RMSE_split_to_2025": rmse_2,
                "N_Full": n_full,
                "N_2019_to_split": n_1,
                "N_split_to_2025": n_2,
            })

rmse_robustness_df = pd.DataFrame(rows).sort_values(["Horizon", "Maturity", "Model"])

# Round for display
for c in ["RMSE_Full", "RMSE_2019_to_split", "RMSE_split_to_2025"]:
    rmse_robustness_df[c] = rmse_robustness_df[c].round(6)

display(rmse_robustness_df)

# Optional: quick check that sample sizes are near-equal for most rows
print("Unique (N_2019_to_split, N_split_to_2025) pairs:",
      sorted(set(zip(rmse_robustness_df["N_2019_to_split"], rmse_robustness_df["N_split_to_2025"])))[:10])
rmse_robustness_df.to_csv("rmse_robustness_df.csv", index=False)



In [ ]:
maturity = "DGS10"
h = 10
key = (maturity, h)

for model in model_names:
    y_pred = model_preds[model][key]
    y_true = model_actuals["RW"][key]

    overlap = y_pred.index.intersection(y_true.index)
    print(model, "pred idx min/max:", y_pred.index.min(), y_pred.index.max(),
          "| true idx min/max:", y_true.index.min(), y_true.index.max(),
          "| overlap:", len(overlap),
          "| pred-only:", len(y_pred.index.difference(y_true.index)),
          "| true-only:", len(y_true.index.difference(y_pred.index)))


# Diebold-Marino Test

In [ ]:
import numpy as np
from scipy.stats import norm

def dm_test(errors_model1, errors_model2, h=1, lag=None, apply_hln=True):
    """
    Diebold–Mariano (DM) test for equal predictive accuracy using squared-error loss,
    with Newey–West HAC variance and optional Harvey–Leybourne–Newbold (HLN) small-sample
    correction.

    Parameters
    ----------
    errors_model1 : array-like
        Forecast errors of model 1 (y - y_hat1), length N.
    errors_model2 : array-like
        Forecast errors of model 2 (y - y_hat2), length N.
    h : int, optional
        Forecast horizon (in periods). Used for HLN correction and (if lag is None)
        for setting the Newey–West truncation lag. Default is 1.
    lag : int, optional
        Newey–West truncation lag for HAC variance of the loss differential.
        If None, lag is set to max(h-1, 0).
    apply_hln : bool, optional
        If True, apply the HLN small-sample correction (Harvey et al., 1997) to the
        DM statistic. Default is True.

    Returns
    -------
    dm_stat : float
        DM statistic (HLN-adjusted if apply_hln=True).
    p_value : float
        Two-sided p-value under asymptotic N(0,1).
    """

    e1 = np.asarray(errors_model1, dtype=float)
    e2 = np.asarray(errors_model2, dtype=float)

    # Drop NaNs in parallel
    mask = np.isfinite(e1) & np.isfinite(e2)
    e1 = e1[mask]
    e2 = e2[mask]

    if e1.shape != e2.shape:
        raise ValueError("Error series must have the same length after NaN removal.")

    N = len(e1)
    if N < 5:
        raise ValueError("Not enough observations for DM test.")

    if h < 1:
        raise ValueError("h must be >= 1.")

    # Squared-error loss and loss differential
    d = (e1 ** 2) - (e2 ** 2)
    d_bar = np.mean(d)

    # Newey–West HAC variance of d_t
    if lag is None:
        lag = max(h - 1, 0)
    if lag < 0:
        raise ValueError("lag must be >= 0.")

    d_centered = d - d_bar
    gamma0 = np.dot(d_centered, d_centered) / N
    var_d = gamma0

    for k in range(1, lag + 1):
        cov = np.dot(d_centered[k:], d_centered[:-k]) / N
        weight = 1.0 - k / (lag + 1)  # Bartlett weight
        var_d += 2.0 * weight * cov

    if var_d <= 0 or not np.isfinite(var_d):
        raise ValueError(f"Non-positive/invalid HAC variance estimate: var_d={var_d}")

    dm_stat = d_bar / np.sqrt(var_d / N)

    # Harvey–Leybourne–Newbold (1997) small-sample correction
    if apply_hln:
        # HLN factor: sqrt((N + 1 - 2h + h(h/N)) / N)
        hln_factor = np.sqrt((N + 1 - 2 * h + (h * (h / N))) / N)
        dm_stat = dm_stat * hln_factor

    # Two-sided p-value under asymptotic N(0,1)
    p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))

    return dm_stat, p_value


In [ ]:
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]

dm_results = []

benchmark_name = "RW"

if benchmark_name not in model_actuals:
    raise ValueError(f"Benchmark model '{benchmark_name}' not loaded.")

# Use RW's actual series as canonical ground truth
rw_actual_series = model_actuals[benchmark_name]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Use RW's actual series as canonical ground truth for this (maturity, h)
        if key not in rw_actual_series:
            continue

        actual = rw_actual_series[key].copy()
        actual.name = "actual"

        # Build a dict of aligned prediction series for all models that exist
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need at least RW + one other model with forecasts
        if benchmark_name not in preds_for_key or len(preds_for_key) < 2:
            continue

        # Compare each non-RW model against RW
        for model_name, pred_series in preds_for_key.items():
            if model_name == benchmark_name:
                continue

            # Pairwise alignment: actual, RW, and the other model
            df_pair = pd.concat(
                [
                    actual,
                    preds_for_key[benchmark_name],  # RW predictions
                    pred_series                     # other model predictions
                ],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_rw    = df_pair["actual"] - df_pair[benchmark_name]
            err_other = df_pair["actual"] - df_pair[model_name]

            # Diebold–Mariano test (RW as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_rw.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  benchmark_name,
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair)
            })

dm_results_df = pd.DataFrame(dm_results)
dm_results_df = dm_results_df.sort_values(["Maturity", "Horizon", "Model_2"])

display(dm_results_df)

In [ ]:
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons      = [1, 5, 10, 30]

dm_results = []

# -------------------------------------------------
# Benchmark: DNS
# -------------------------------------------------
benchmark_name = "DNS"
models_to_compare = ["Ridge", "XGBoost"]   # you can add "RW" here if you change your mind

if benchmark_name not in model_actuals:
    raise ValueError(f"Benchmark model '{benchmark_name}' not loaded.")

# Use DNS' actual series as canonical ground truth (should be identical across models)
dns_actual_series = model_actuals[benchmark_name]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Check that we have actuals for this maturity/horizon
        if key not in dns_actual_series:
            continue

        # Actual LEVEL yields at forecast dates
        actual = dns_actual_series[key].copy()
        actual.name = "actual"

        # -------------------------------------------------
        # Collect prediction series for this (maturity, h)
        # -------------------------------------------------
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need benchmark + at least one comparison model
        if benchmark_name not in preds_for_key:
            continue

        # -------------------------------------------------
        # Compare each model in models_to_compare vs DNS
        # -------------------------------------------------
        for model_name in models_to_compare:
            if model_name not in preds_for_key:
                continue
            if model_name == benchmark_name:
                continue

            pred_bench = preds_for_key[benchmark_name]
            pred_other = preds_for_key[model_name]

            # Pairwise alignment: actual, DNS, and the other model
            df_pair = pd.concat(
                [actual, pred_bench, pred_other],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_bench = df_pair["actual"] - df_pair[benchmark_name]   # DNS errors
            err_other = df_pair["actual"] - df_pair[model_name]       # other model

            # Diebold–Mariano test (DNS as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_bench.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  benchmark_name,
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair),
            })

# -------------------------------------------------
# Build results DataFrame
# -------------------------------------------------
dm_dns_benchmark_df = (
    pd.DataFrame(dm_results)
    .sort_values(["Maturity", "Horizon", "Model_2"])
    .reset_index(drop=True)
)

display(dm_dns_benchmark_df)


# Root-Mean-Squared Error

In [ ]:
# ----------------------------------------------------
# RMSE comparison table across models and horizons
# ----------------------------------------------------

rmse_tables = []

for model_name, df in model_metrics.items():
    if "RMSE" not in df.columns:
        print(f"⚠️ No RMSE column found for {model_name}, skipping.")
        continue

    rmse_tables.append(
        df[["RMSE"]].rename(columns={"RMSE": model_name})
    )

rmse_compare_df = pd.concat(rmse_tables, axis=1).sort_index()

display(rmse_compare_df)
